In [1]:
from cellgrn.main import normalzie_rna,parse_edges,compute_all_cells_grn,summarize_grn,format_sample_grn,format_celltype_grn,compute_tf_gene_score
import numpy as np
import pandas as pd
import os
import anndata as ad
from scipy import sparse
import pickle



/home/shaliu_fu/miniconda3/envs/sparsegrn/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
cell_meta = pd.read_csv("/home/shaliu_fu/multireg/cellGRN/data/spatial_brain/metadata.csv")

input_rna = ad.read_h5ad(f"/home/shaliu_fu/multireg/cellGRN/data/spatial_brain/rna_count.h5ad")
input_atac = ad.read_h5ad(f"/home/shaliu_fu/multireg/cellGRN/data/spatial_brain/atac_count.h5ad")
cell_meta.index = cell_meta['barcode']

In [3]:
# all_tf = [i.rstrip() for i in open("/home/shaliu_fu/multireg/multigrn/db/all_hg_TF.txt")] 

In [3]:
# cell_meta

In [4]:
input_gene = input_rna.var.index.values
input_peak = input_atac.var.index.values
# input_tf = list(set(input_gene) & set(all_tf))
cell_types = cell_meta['Sample']

In [5]:
# input_tf = cand_df[cand_df['group_subtype']!="Peak-Gene"]['feature1'].unique()

In [8]:
# input_df2
input_df1 = pd.DataFrame(input_rna.X,index=input_rna.obs.index.values,columns=input_rna.var.index.values)

In [12]:
rna_data1,rna_data2 = normalzie_rna(input_df1)


In [15]:
a=rna_data1['Eomes']
a[a>0]

E11_0-S1_CTGATGGTCTTGCGCC-1    0.004024
E11_0-S1_CCGAGAATCTTGCGCC-1    0.002646
E11_0-S1_TTGACTTCACTTAACC-1    0.007042
E11_0-S1_ACTCAATACCAGTTCC-1    0.014286
E13_5-S1_CCGAGAATGCAGGTCC-1    0.012987
                                 ...   
E18_5-S1_TAGATCTAGATACGGA-1    0.003367
E18_5-S1_TTATTCATACTATGCA-1    0.004662
E18_5-S1_AATTAAGAGATACGGA-1    0.003610
E18_5-S1_AATTAAGATGCGGACC-1    0.003185
E18_5-S1_ATCATATTACTATGCA-1    0.003906
Name: Eomes, Length: 67, dtype: float64

In [7]:
# input_peaks

In [16]:
soft = "scenic2"


outdir = f"../output/res_spatial_brain_{soft}/"
os.system(f"mkdir -p {outdir}")



input_df1 = pd.DataFrame(input_rna.X,index=input_rna.obs.index.values,columns=input_rna.var.index.values)
# peak_rename = [i.replace(":","-") for i in input_atac.var.index.values]
peak_rename = [i.replace("-",":",1) for i in input_atac.var.index.values]
# peak_rename = input_peak
input_df2 = pd.DataFrame(input_atac.X,index=input_atac.obs.index.values,columns=peak_rename)


cand_df = pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN/data/spatial_brain/{soft}_grn.csv",header=0)

input_genes = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/spatial_brain/{soft}_genes.txt")]
input_peaks = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN/data/spatial_brain/{soft}_peaks.txt")]

# input_peaks = [i.replace("-",":",1) for i in input_peaks]
input_df1 = input_df1[input_genes]
input_df2 = input_df2[input_peaks]


rna_data1,rna_data2 = normalzie_rna(input_df1)
atac_data = input_df2.copy()

input_tf = cand_df[cand_df['group_subtype']!="Peak-Gene"]['feature1'].unique()

input_tfs = [tf for tf in input_tf if tf in input_genes]
tf_data1 = rna_data1[input_tfs].copy()
tf_data2 = rna_data2[input_tfs].copy()

edges_idx,edges_name = parse_edges(cand_df, input_tfs, input_genes, input_peaks)



True

In [18]:
grn_scale2 = compute_all_cells_grn(tf_data2, rna_data2, atac_data,edges_idx, edges_name,
    input_tfs, input_genes, input_peaks)

In [ ]:
# sel_names  = ['Peak-Gene_chr9:118455364-118455864_Eomes',
#  'Peak-Gene_chr9:118228695-118229195_Eomes',
#  'Peak-Gene_chr9:118318700-118319200_Eomes',
#  'Peak-Gene_chr9:118246831-118247331_Eomes']

In [ ]:

grn_scale2 = compute_all_cells_grn(tf_data2, rna_data2, atac_data,edges_idx, edges_name,
    input_tfs, input_genes, input_peaks)

with open(f"{outdir}/{soft}_cell_grn.pkl", "wb") as f:
    pickle.dump(grn_scale2, f)

sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)


tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)




tf_gene_res_scale2.to_csv(os.path.join(outdir, "tf_gene_sample_scale2.csv"), index=False)
tf_peak_res_scale2.to_csv(os.path.join(outdir, "tf_peak_sample_scale2.csv"), index=False)
gene_peak_res_scale2.to_csv(os.path.join(outdir, "gene_peak_sample_scale2.csv"), index=False)

tf_gene_ct_res_scale2.to_csv(os.path.join(outdir, "tf_gene_celltype_scale2.csv"), index=False)
tf_peak_ct_res_scale2.to_csv(os.path.join(outdir, "tf_peak_celltype_scale2.csv"), index=False)
gene_peak_ct_res_scale2.to_csv(os.path.join(outdir, "gene_peak_celltype_scale2.csv"), index=False)